# atari_200m — DQN vs. Random on Atari Pong (200M frames)

Overlays the two components (`random_pong`, `dqn_pong`) as episodic-return
curves, same as the `atari_20m` notebook. This experiment runs a **single seed**,
so the bootstrap-CI band collapses to the mean line. (Atari is cluster-scale —
this notebook expects results produced elsewhere; see the experiment README.)

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from experiment import load_curves, load_runs
from experiment.plotting import seed_grids_for, plot_mean_ci, style


def grid_for(component_db, n=500):
    """A shared [0, T] timestep grid from any run's curve length (T = TOTAL_TIMESTEPS)."""
    df = load_runs(component_db, write_csv=False)
    T = len(load_curves(component_db, df["run_id"][0])["reward"])
    return np.linspace(0, T, n)


In [ ]:
HERE = Path.cwd()
RESULTS = HERE / "results" if (HERE / "results").exists() else Path("experiments/atari_200m/results")

# Both components run 50M steps, so they share one grid.
GRID = grid_for(RESULTS / "dqn_pong.db")

series = [
    ("Random Agent", "tab:red", RESULTS / "random_pong.db"),
    ("DQN", "tab:blue", RESULTS / "dqn_pong.db"),
]

fig, ax = plt.subplots(figsize=(9, 6))   # 2:3 height:width
for label, color, comp_dir in series:
    stack = seed_grids_for(comp_dir, GRID)
    n = stack.shape[0]
    plot_mean_ci(ax, GRID, stack, f"{label} (n={n})", color)

ax.set_title("DQN vs. Random Agent on Atari Pong, 200M frames  (mean ± 95% bootstrap CI)")
ax.legend(loc="lower right", frameon=False)
style(ax, ylim=(-21, 21))   # Pong score range
fig.tight_layout()
plt.show()


In [ ]:
PLOTS_DIR = RESULTS.parent / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
fig.savefig(PLOTS_DIR / "atari_random_vs_dqn.pdf", bbox_inches="tight")
print(f"saved plot to {PLOTS_DIR / 'atari_random_vs_dqn.pdf'}")
